In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2_sup")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_16_9_0,0.999999,0.992942,0.999993,0.999998,0.999995,0.222811,1574.291995,0.501092,0.043574,0.272325,0.004054,0.472029,1.000001,0.492124,89.002863,141.414524,"Hidden Size=[4, 5], regularizer=0.05, learning..."
1,model_16_8_24,0.999999,0.992952,0.999991,0.999764,0.999989,0.234126,1572.080839,0.605504,0.845555,0.725529,0.004051,0.483866,1.000001,0.504465,88.903792,141.315453,"Hidden Size=[4, 5], regularizer=0.05, learning..."
2,model_16_8_23,0.999999,0.992953,0.999991,0.999762,0.999989,0.234778,1571.959717,0.601823,0.851807,0.726876,0.004201,0.484539,1.000001,0.505167,88.898228,141.309888,"Hidden Size=[4, 5], regularizer=0.05, learning..."
3,model_16_8_21,0.999999,0.992952,0.999990,0.999762,0.999989,0.238227,1572.192230,0.617823,0.852959,0.735400,0.004509,0.488085,1.000001,0.508864,88.869063,141.280723,"Hidden Size=[4, 5], regularizer=0.05, learning..."
4,model_16_8_22,0.999999,0.992951,0.999990,0.999761,0.999989,0.238439,1572.291267,0.619673,0.856389,0.738031,0.004372,0.488302,1.000001,0.509090,88.867284,141.278944,"Hidden Size=[4, 5], regularizer=0.05, learning..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3762,model_33_8_8,0.996258,0.992857,0.999615,0.999506,0.999607,834.781065,1593.307847,98.802531,21.373409,60.087922,0.219497,28.892578,1.001109,30.122595,196.545661,324.527623,"Hidden Size=[8, 8], regularizer=0.2, learning_..."
3769,model_33_8_7,0.996180,0.992803,0.999605,0.999444,0.999590,852.117626,1605.386787,101.476884,24.042432,62.759707,0.221786,29.191054,1.001132,30.433777,196.504551,324.486512,"Hidden Size=[8, 8], regularizer=0.2, learning_..."
3776,model_33_8_6,0.996100,0.992746,0.999591,0.999372,0.999568,869.998407,1617.998641,105.022501,27.152841,66.087671,0.224189,29.495735,1.001156,30.751430,196.463017,324.444979,"Hidden Size=[8, 8], regularizer=0.2, learning_..."
3783,model_33_8_5,0.996017,0.992687,0.999573,0.999290,0.999542,888.432943,1631.173210,109.456653,30.724158,70.089872,0.226612,29.806592,1.001180,31.075521,196.421082,324.403043,"Hidden Size=[8, 8], regularizer=0.2, learning_..."
